# Bài toán 3 — Thiết lập và xây dựng mô hình

## 5. Thiết lập thực nghiệm
Giữ ba lượt validation theo thời gian, mỗi lượt dự báo 548 ngày. Tập phát triển kết thúc 01/07/2021; tập đánh giá hồi cứu từ 02/07/2021 đến 31/12/2022 (548 ngày). Mọi đặc trưng, hệ số và hiệu chỉnh ngoài mẫu chỉ sử dụng dữ liệu trước origin. Không dùng nhãn test để chọn phương pháp. Kiến trúc đã được khảo sát trước đây trên lịch sử dài hơn nên đánh giá hồi cứu không độc lập hoàn toàn đối với lựa chọn kiến trúc; tham số được khóa trước quy trình này.

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sales_path = Path("data/raw/sales.csv")
start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT", Path.cwd())).resolve()
project_root = next(
    (folder for folder in (start, *start.parents) if (folder / sales_path).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/sales.csv. Hãy đặt notebook trong dự án "
        "hoặc trỏ RETAIL_ANALYTICS_ROOT về thư mục gốc."
    )

targets = ["Revenue", "COGS"]
columns = ["Date", *targets]
sales_daily = pd.read_csv(project_root / sales_path, usecols=columns, dtype="string")[columns]
sales_daily["Date"] = pd.to_datetime(sales_daily["Date"], format="%Y-%m-%d", errors="raise")
if sales_daily["Date"].isna().any():
    raise ValueError("Date có giá trị thiếu.")

sales_daily[targets] = sales_daily[targets].apply(pd.to_numeric, errors="raise").astype(float)
sales_daily = sales_daily.sort_values("Date", ignore_index=True)

history_start = pd.Timestamp("2012-07-04")
history_end = pd.Timestamp("2022-12-31")
expected_dates = pd.date_range(history_start, history_end, freq="D")
if not pd.DatetimeIndex(sales_daily["Date"]).equals(expected_dates):
    raise ValueError("Lịch sử có ngày thiếu, trùng hoặc ngoài khoảng đã xác nhận.")
if not np.isfinite(sales_daily[targets].to_numpy(dtype=float)).all():
    raise ValueError("Revenue hoặc COGS có giá trị thiếu hoặc không hữu hạn.")


horizon = 548
n_folds = 3
dev_start = history_start
dev_end = sales_daily["Date"].iloc[-horizon - 1]
sales_dev = sales_daily.set_index("Date").loc[dev_start:dev_end].copy()



one_day = pd.Timedelta(days=1)

test_dates = sales_daily.loc[sales_daily["Date"] > dev_end, "Date"]
if len(test_dates) != horizon:
    raise ValueError("Tập test phải có đúng 548 ngày.")

rows = []
for fold in range(1, n_folds + 1):
    val_end = dev_end - pd.Timedelta(days=(n_folds - fold) * horizon)
    val_start = val_end - pd.Timedelta(days=horizon - 1)
    train_end = val_start - one_day

    train = sales_dev.loc[:train_end]
    validation = sales_dev.loc[val_start:val_end]
    if train.empty or len(validation) != horizon:
        raise ValueError(f"Fold {fold} không đủ dữ liệu.")

    rows.append({
        "Fold": fold,
        "Train từ": train.index.min().date(),
        "Train đến": train_end.date(),
        "Số ngày train": len(train),
        "Validation từ": val_start.date(),
        "Validation đến": val_end.date(),
        "Số ngày validation": len(validation),
    })

folds = pd.DataFrame(rows)
display(folds)
print(f"Test cuối: {test_dates.min():%d/%m/%Y}–{test_dates.max():%d/%m/%Y} ({len(test_dates)} ngày)")
assert (folds["Train đến"] < folds["Validation từ"]).all()
assert pd.to_datetime(folds["Validation đến"]).max() < test_dates.min()
assert dev_end == pd.Timestamp("2021-07-01")
assert len(sales_dev) == 3285
assert pd.DatetimeIndex(test_dates).equals(pd.date_range("2021-07-02", "2022-12-31"))

,Fold,Train từ,Train đến,Số ngày train,Validation từ,Validation đến,Số ngày validation
0,1,2012-07-04,2016-12-30,1641,2016-12-31,2018-07-01,548
1,2,2012-07-04,2018-07-01,2189,2018-07-02,2019-12-31,548
2,3,2012-07-04,2019-12-31,2737,2020-01-01,2021-07-01,548


Test cuối: 02/07/2021–31/12/2022 (548 ngày)


## 6. Xây dựng nhiều phương pháp
Giữ 10 phương pháp đối chứng ban đầu. Từ mô hình mùa vụ mức gốc cố định và hiệu ứng thứ, phát triển một phương pháp Hybrid kết hợp phân rã doanh thu, mùa vụ và phân khúc. Tất cả so sánh trên cùng ba lượt validation.

In [2]:
import sys
from time import perf_counter
import warnings

import statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | NumPy {np.__version__} | statsmodels {statsmodels.__version__}")

contest_name = "Baseline cuộc thi"
fixed_name = "Mùa vụ, mức gốc cố định"
mean_name = "Trung bình 28 ngày"
validation_csv_label = "Phương pháp được chọn"  # Nhãn CSV tương thích với bản trước.


def repeat_level(level, dates):
    return pd.DataFrame(
        np.tile(level.to_numpy(dtype=float), (len(dates), 1)),
        index=dates, columns=level.index,
    )

Python 3.11.7 | pandas 2.2.3 | NumPy 2.4.6 | statsmodels 0.15.0


### 6.1. Naive

Lặp lại giá trị cuối cùng của tập train cho toàn bộ kỳ dự báo. Đây là mốc đơn giản nhất.

In [3]:
def predict_naive(train, dates):
    return repeat_level(train.iloc[-1], dates)

### 6.2. Trung bình 28 ngày

Dùng trung bình của 28 ngày cuối tập train làm mức dự báo cố định.

In [4]:
def predict_mean_28(train, dates):
    return repeat_level(train.tail(28).mean(), dates)

### 6.3. Theo thứ trong tuần

Dùng trung bình ứng với từng thứ trong 28 ngày cuối. Thứ của ngày dự báo được xác định từ lịch.

In [5]:
def predict_weekday(train, dates):
    recent = train.tail(28)
    weekday = recent.groupby(recent.index.dayofweek).mean()
    forecast = weekday.reindex(dates.dayofweek)
    forecast.index = dates
    return forecast

### 6.4. ETS (A,N,N)

Làm trơn mũ đơn giản để ước lượng mức của chuỗi. Mô hình dùng sai số cộng, không có thành phần xu hướng hoặc mùa vụ.

In [6]:
def predict_ets(train, dates):
    forecast = pd.DataFrame(index=dates, columns=train.columns, dtype=float)
    for target in train.columns:
        history = train[target].astype(float).asfreq("D") / 1_000_000
        fitted = SimpleExpSmoothing(history, initialization_method="estimated").fit()
        estimate = fitted.forecast(len(dates))
        pd.testing.assert_index_equal(estimate.index, dates, check_names=False)
        forecast[target] = estimate.to_numpy(dtype=float) * 1_000_000
    return forecast

### 6.5. ARIMA(1,1,1)

Lấy sai phân một lần, sau đó khớp thành phần tự hồi quy bậc 1 và trung bình trượt bậc 1. Dùng cùng cấu hình cho từng chỉ tiêu và từng lượt.

In [7]:
def predict_arima(train, dates):
    forecast = pd.DataFrame(index=dates, columns=train.columns, dtype=float)
    for target in train.columns:
        history = train[target].astype(float).asfreq("D") / 1_000_000
        with warnings.catch_warnings():
            # Statsmodels dùng điểm khởi tạo bằng 0 khi ước lượng ban đầu không phù hợp.
            warnings.filterwarnings(
                "ignore", message=r"Non-(stationary|invertible) starting .*", category=UserWarning,
            )
            fitted = ARIMA(history, order=(1, 1, 1)).fit()
        if not fitted.mle_retvals["converged"]:
            raise RuntimeError(f"ARIMA không hội tụ cho {target}, mốc {train.index[-1]:%d/%m/%Y}.")
        estimate = fitted.forecast(len(dates))
        pd.testing.assert_index_equal(estimate.index, dates, check_names=False)
        forecast[target] = estimate.to_numpy(dtype=float) * 1_000_000
    return forecast

### 6.6. Hồi quy tuyến tính

Dùng số ngày kể từ ngày đầu tập train làm đặc trưng và khớp một đường xu hướng riêng cho từng chỉ tiêu.

In [8]:
def predict_trend(train, dates):
    origin = train.index[0]
    train_days = (train.index - origin).days.to_numpy()
    future_days = (dates - origin).days.to_numpy()
    slope, intercept = np.polyfit(train_days, train.to_numpy(dtype=float), 1)
    forecast = pd.DataFrame(
        intercept + future_days[:, None] * slope, index=dates, columns=train.columns,
    )
    return forecast

### 6.7. Hồi quy tuyến tính + mùa vụ

Tính phần dư của đường xu hướng trên tập train, lấy trung bình theo cặp tháng–ngày rồi cộng vào dự báo tương ứng.

In [9]:
def predict_trend_with_seasonality(train, dates):
    trend = predict_trend(train, train.index.append(dates))
    residuals = train - trend.loc[train.index]
    seasonal = residuals.groupby([train.index.month, train.index.day]).mean()
    month_day = pd.MultiIndex.from_arrays([dates.month, dates.day])
    adjustment = seasonal.reindex(month_day).fillna(0.0)
    adjustment.index = dates
    return trend.loc[dates] + adjustment

### 6.8. Baseline cuộc thi

Dùng mức ngày của năm đầy đủ gần nhất, hệ số tăng trưởng hằng năm và hệ số mùa vụ theo tháng–ngày. Tất cả được tính lại từ tập train của từng lượt.

In [10]:
def predict_contest_baseline(train, forecast_dates):
    years = train.index.year
    days_observed = train.groupby(years).size()
    days_in_year = pd.to_datetime(
        days_observed.index.astype(str) + "-12-31"
    ).dayofyear
    complete_years = days_observed.index[days_observed.eq(days_in_year)]

    # Tính mức gốc và tăng trưởng từ các năm đủ dữ liệu.
    annual_totals = train.groupby(years).sum().loc[complete_years]
    base_year = complete_years[-1]
    base_level = annual_totals.loc[base_year] / days_observed.loc[base_year]
    growth = (annual_totals.iloc[-1] / annual_totals.iloc[0]) ** (
        1 / (base_year - complete_years[0])
    )

    # Tính hệ số mùa vụ theo cặp tháng–ngày từ tập train.
    normalized = train.div(train.groupby(years).transform("mean"))
    seasonal = normalized.groupby([train.index.month, train.index.day]).mean()
    month_day = pd.MultiIndex.from_arrays([forecast_dates.month, forecast_dates.day])
    seasonal_factors = seasonal.reindex(month_day).fillna(1.0)
    seasonal_factors.index = forecast_dates

    years_ahead = (forecast_dates.year - base_year).to_numpy()
    growth_factors = growth.to_numpy() ** years_ahead[:, None]
    return seasonal_factors.mul(growth_factors).mul(base_level).round(2)

### 6.9. Mùa vụ, mức gốc cố định

Giữ mức trung bình ngày của năm đầy đủ gần nhất, sau đó nhân với hệ số mùa vụ theo tháng–ngày học từ các quan sát trong tập train.

In [11]:
def predict_fixed_level_seasonal(train, forecast_dates):
    # Lấy trung bình ngày của năm đầy đủ gần nhất làm mức gốc.
    years = train.index.year
    days_observed = train.groupby(years).size()
    days_in_year = pd.to_datetime(
        days_observed.index.astype(str) + "-12-31"
    ).dayofyear
    complete_years = days_observed.index[days_observed.eq(days_in_year)]
    base_year = complete_years[-1]
    base_level = train.loc[str(base_year)].mean()

    # Tính hệ số mùa vụ cho từng cặp tháng–ngày từ tập train.
    normalized = train.div(train.groupby(years).transform("mean"))
    seasonal = normalized.groupby([train.index.month, train.index.day]).mean()
    month_day = pd.MultiIndex.from_arrays([forecast_dates.month, forecast_dates.day])
    factors = seasonal.reindex(month_day).fillna(1.0)
    factors.index = forecast_dates

    return factors.mul(base_level).round(2)

### 6.10. Mùa vụ mức gốc cố định có hiệu ứng thứ trong tuần
Mô hình bổ sung hệ số theo thứ trong tuần vào dự báo mùa vụ ở mục 6.9. Hệ số được tính riêng cho Revenue và COGS từ tập train của từng lượt validation.

1. **Chuẩn hóa theo năm.** Chia giá trị của mỗi ngày cho mức trung bình ngày của năm đó trong tập train.
2. **So sánh cùng ngày trên lịch.** Với từng ngày, chia giá trị đã chuẩn hóa cho mức trung bình của cùng cặp tháng–ngày ở các năm còn lại trong tập train. Bước này giúp tách ảnh hưởng của ngày trong năm khi ước lượng hiệu ứng theo thứ.
3. **Tính hệ số theo thứ.** Lấy trung vị các tỷ số theo từng thứ, rồi chia mỗi hệ số cho trung bình của bảy hệ số để chúng có trung bình bằng 1.

Dự báo của mô hình đối chứng này bằng **dự báo mùa vụ ở mục 6.9 × hệ số của thứ tương ứng với ngày dự báo**. Ví dụ, hệ số thứ Bảy bằng 1,08 nghĩa là dự báo cho ngày thứ Bảy được điều chỉnh tăng 8% so với dự báo mùa vụ gốc.

In [12]:
def predict_fixed_level_with_weekday(train, forecast_dates):
    base_forecast = predict_fixed_level_seasonal(train, forecast_dates)
    years = train.index.year
    normalized = train.astype(float).div(train.groupby(years).transform("mean"))
    ratios = []

    for year in years.unique():
        current_year = normalized.loc[years == year]
        other_years = normalized.loc[years != year]

        seasonal = other_years.groupby(
            [other_years.index.month, other_years.index.day]
        ).mean()

        dates = current_year.index
        month_day = pd.MultiIndex.from_arrays([dates.month, dates.day])
        reference = seasonal.reindex(month_day).set_axis(dates)
        ratios.append(current_year.div(reference))

    ratios = pd.concat(ratios)
    weekday_factors = ratios.groupby(ratios.index.dayofweek).median()
    weekday_factors = weekday_factors.div(weekday_factors.mean())

    adjustment = weekday_factors.reindex(forecast_dates.dayofweek)
    adjustment.index = forecast_dates
    return base_forecast.mul(adjustment).round(2)

### 6.11. Mô hình đề xuất: Hybrid Ensemble kết hợp phân rã doanh thu và phân khúc sản phẩm
Mô hình 6.10 giữ quy luật mùa vụ và thứ, nhưng mức gốc cố định chưa trực tiếp mô tả biến động số dòng hàng và giá trị mỗi dòng. Notebook 01 phát hiện thay đổi mức kinh doanh từ 2019, quy luật chẵn/lẻ và khác biệt giữa tám phân khúc. Đây là căn cứ để xây dựng mô hình thích nghi, chưa phải bằng chứng rằng mô hình mới tốt hơn.

Với ngày $t$, $R_t=N_t\bar r_t$, $C_t=N_t\bar c_t$, trong đó $N_t$ là số dòng hàng, $\bar r_t$ và $\bar c_t$ là doanh thu và giá vốn trung bình mỗi dòng. Nhánh cơ sở xấp xỉ bằng số dòng, lượng trung bình, đơn giá tham chiếu và tỷ lệ giá bán; nhánh phân khúc dự báo riêng số dòng và giá trị mỗi dòng rồi tổng hợp.

**Quy trình:** dự báo thành phần → phối hợp mùa vụ và CatBoost → phục hồi có điều kiện → hiệu chỉnh sai số ngoài mẫu → xu hướng giảm dần và chu kỳ hai năm → nhánh tám phân khúc. Đây là các thành phần nội bộ của một mô hình, không phải các phiên bản độc lập.

#### a. Chuẩn bị dữ liệu giao dịch
Chỉ lấy đơn hàng đến 01/07/2021 trước khi ghép dòng hàng và sản phẩm; cắt lại theo từng origin. Chỉ dùng tháng/năm đầy đủ cho đặc trưng tổng hợp. Danh mục giá tham chiếu và giá vốn được xem là tĩnh theo dữ liệu cung cấp.

In [13]:
from scipy.ndimage import gaussian_filter1d
from scipy.sparse import hstack
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.preprocessing import OneHotEncoder
from threadpoolctl import threadpool_limits
from catboost import CatBoostRegressor

TARGETS = targets
hybrid_name = "Hybrid Ensemble phân rã và phân khúc"
weekday_name = "Mùa vụ mức gốc cố định + hiệu ứng thứ"

def read_raw(name, **kwargs):
    return pd.read_csv(project_root / "data/raw" / name, **kwargs)

orders = read_raw("orders.csv", usecols=["order_id", "order_date"], parse_dates=["order_date"])
orders = orders.loc[orders.order_date.between(dev_start, dev_end)]
items = read_raw("order_items.csv", usecols=["order_id", "product_id", "quantity", "unit_price"])
products = read_raw("products.csv", usecols=["product_id", "segment", "price", "cogs"])
assert orders.order_id.is_unique and products.product_id.is_unique
items = items.loc[items.order_id.isin(orders.order_id)]
lines = items.merge(orders, on="order_id", validate="many_to_one").merge(products, on="product_id", validate="many_to_one")
assert len(lines) == len(items)
lines["Revenue"] = lines.quantity * lines.unit_price
lines["COGS"] = lines.quantity * lines.cogs
lines = lines.assign(base=lines.price, actual=lines.unit_price, cost=lines.cogs, N=1, revenue=lines.Revenue)
daily = lines.groupby("order_date").agg(lines=("quantity", "size"), quantity=("quantity", "mean"), price=("price", "mean"), cost=("cogs", "mean"), actual_price=("unit_price", "mean"), Revenue=("Revenue", "sum"), COGS=("COGS", "sum"))
daily["price_factor"] = daily.actual_price / daily.price
assert daily.index.equals(sales_dev.index)
np.testing.assert_allclose(daily[TARGETS], sales_dev[TARGETS], atol=.011, rtol=0)
daily[TARGETS] = sales_dev[TARGETS].to_numpy()
panel = lines.groupby(["order_date", "segment"])[["base", "actual", "cost", "N", "quantity", "revenue"]].sum().unstack(fill_value=0)
seg_daily = lines.groupby(["order_date", "segment"])[["N", "Revenue", "COGS"]].sum().rename(columns={"N":"lines"}).unstack(fill_value=0)
assert panel.index.equals(daily.index) and seg_daily.index.equals(daily.index)
assert len(panel["N"].columns) == 8
np.testing.assert_allclose(seg_daily["Revenue"].sum(axis=1), daily.Revenue, atol=.011, rtol=0)
np.testing.assert_allclose(seg_daily["COGS"].sum(axis=1), daily.COGS, atol=.011, rtol=0)
print(f"Đối soát {len(daily):,} ngày, {len(lines):,} dòng hàng, {len(panel['N'].columns)} phân khúc.")
del orders, items, products, lines

def complete_months(frame, how="mean"):
    grouped = frame.resample("MS")
    result = getattr(grouped, how)()
    count = grouped.size()
    return result.loc[count == count.index.days_in_month]


Đối soát 3,285 ngày, 661,176 dòng hàng, 8 phân khúc.


#### b. Dự báo thành phần và mức kinh doanh thích nghi
Poisson Regression phù hợp với số dòng hàng; HistGradient Boosting bổ sung quan hệ phi tuyến (65%–35%). Ridge trên log dùng cho đơn giá tham chiếu và giá vốn. Nhánh xu hướng tháng kết hợp spline, chu kỳ dài và tín hiệu tăng trưởng lịch sử.

In [14]:
def design(dates, training_size):
    """Hiệu ứng lịch có co rút. Xu hướng chậm được ước tính riêng."""
    month = dates.month.to_numpy()
    day = dates.day.to_numpy()
    week = dates.dayofweek.to_numpy()
    end = dates.days_in_month.to_numpy() - day
    odd = dates.year.to_numpy() % 2
    key = month * 100 + day
    md_phase = np.where(day < 9, day, np.where(end < 8, -1 - end, 0))
    blocks = []
    def add(weight, *features):
        frame = np.column_stack(features)
        # Mỗi tổ hợp lịch tương ứng với một biến chỉ báo.
        tokens = pd.DataFrame(frame).astype(str).agg("|".join, axis=1).to_numpy()[:, None]
        encoder = OneHotEncoder(handle_unknown="ignore")
        encoder.fit(tokens[:training_size])
        blocks.append(encoder.transform(tokens) * weight)
    add(2, month)
    add(2, day)
    add(2, end)
    add(2, week)
    add(1, month, odd)
    add(.9, month, md_phase)
    add(.65, key)
    add(.48, key, odd)
    add(.4, week, md_phase)
    add(.4, month, week)
    year_phase = dates.year.to_numpy() % 4
    add(.85, month, week, year_phase)
    add(.65, (month - 1) // 3, week, year_phase)
    add(.4, week, year_phase)
    add(.6, month, md_phase, odd)
    add(.35, month, dates.is_leap_year)
    # Cho phép mùa vụ gần đây khác những năm đầu dữ liệu.
    add(.3, dates.year >= 2019, month)
    return hstack(blocks, format="csr")

In [15]:
def forecast_lines(past, dates):
    n = len(past)
    calendar = past.index.append(dates)
    indicators = design(calendar, n)
    month = calendar.month.to_numpy()
    day = calendar.day.to_numpy()
    tree_features = np.column_stack([
        month, day, calendar.days_in_month - day, calendar.dayofweek,
        calendar.year % 2, calendar.year % 4, month * 32 + day,
        calendar.is_leap_year, (month - 1) // 3, calendar.dayofyear
    ])
    y = past.lines.to_numpy()
    weights = .5 ** ((n - 1 - np.arange(n)) / (365.25 * 4))
    weights[past.index.year < 2013] = 0
    annual_mean = past.lines.groupby(past.index.year).transform("mean").to_numpy()
    year_size = past.lines.groupby(past.index.year).transform("size")
    first_pass_weights = weights * (year_size.to_numpy() == np.where(past.index.is_leap_year, 366, 365))
    first_pass = PoissonRegressor(alpha=.0006, max_iter=600, tol=3e-6)
    first_pass.fit(indicators[:n], y / annual_mean, sample_weight=first_pass_weights)
    adjusted = y / first_pass.predict(indicators[:n])
    level = gaussian_filter1d(adjusted, sigma=65, mode="nearest", truncate=3)
    level[:90] = np.mean(adjusted[:140])
    level[-50:] = np.mean(adjusted[-105:])
    relative_volume = y / level

    glm = PoissonRegressor(alpha=.00038, max_iter=600, tol=3e-6)
    glm.fit(indicators[:n], relative_volume, sample_weight=weights)
    tree = HistGradientBoostingRegressor(loss="poisson", learning_rate=.06, max_iter=240,
             max_leaf_nodes=26, min_samples_leaf=15, l2_regularization=1, random_state=42)
    tree.fit(tree_features[:n], relative_volume, sample_weight=weights)
    shape = .65 * glm.predict(indicators) + .35 * tree.predict(tree_features)

    historical_ratio = pd.Series(relative_volume / shape[:n], index=past.index).clip(.3, 3.)
    monthly = complete_months(historical_ratio).tail(36)
    by_month = monthly.groupby(monthly.index.month).mean().to_numpy()
    by_month /= by_month.mean()
    smoothed = .25*np.roll(by_month, 1) + .5*by_month + .25*np.roll(by_month, -1)
    recent_shape = shape * (1 + .7 * (smoothed[month-1] - 1))

    def project(profile, use_recent_months):
        weekly = (y / profile[:n])[-728:].reshape(-1, 7).mean(axis=1)
        recent = np.average(weekly[-26:], weights=np.arange(1, 27))
        prior = np.average(weekly[-78:-52], weights=np.arange(1, 27))
        latest = weekly[-9:].mean()
        annual_change = np.clip(np.log(recent / prior), -.30, .30)
        half_year_change = np.clip(2*np.log(latest / weekly[-35:-26].mean()), -.30, .30)
        rate = .6*annual_change + .4*half_year_change
        if use_recent_months:
            latest = np.average(weekly[-16:], weights=np.arange(1, 17))
            rate *= .8
        horizon = np.arange(1, len(dates)+1) / 365.25
        return profile[n:] * latest * np.exp(rate * (1 - np.exp(-horizon)))

    return (project(shape, False) + project(recent_shape, True)) / 2

In [16]:
def forecast_prices(past, dates):
    idx = past.index.append(dates)
    n = len(past)
    time = (idx - idx[n - 1]).days.to_numpy() / 365.25
    columns = [time]
    for key, scale in [(idx.month, 1.), (idx.month*2 + idx.year%2, .65),
                        (idx.month*32 + idx.day, .22), (idx.day, .32),
                        (idx.dayofweek, .3), (idx.month*10 + np.minimum(9, idx.day//4), .5)]:
        columns.extend((pd.get_dummies(key).to_numpy(dtype=float)*scale).T)
    features = np.column_stack(columns)
    model = Ridge(alpha=1.4)
    target = np.log(past[["price","cost"]].to_numpy())
    model.fit(features[:n], target, sample_weight=.5**(-time[:n]/3.5))
    fitted = model.predict(features)
    residual = target - fitted[:n]
    correction = np.average(residual[-180:], axis=0, weights=np.arange(1,181))
    smear = np.mean(np.exp(residual[-365:]),axis=0) / np.exp(np.mean(residual[-365:],axis=0))
    return np.exp(fitted[n:] + correction) * smear

def forecast_components(past, dates):
    dates = pd.DatetimeIndex(dates)
    assert past.index.max() < dates.min()
    count = forecast_lines(past, dates)
    unit_values = forecast_prices(past, dates)
    ratio = past.price_factor.loc["2013":]
    typical = ratio.groupby([ratio.index.year%2, ratio.index.month, ratio.index.day]).median()
    requested = pd.MultiIndex.from_arrays([dates.year%2, dates.month, dates.day])
    unit_values[:, 0] *= typical.reindex(requested).fillna(1.).to_numpy()
    quantity = np.average(past.quantity, weights=past.lines)
    return pd.DataFrame(count[:,None] * quantity * unit_values, index=dates, columns=["Revenue","COGS"])

In [17]:
def curve(frame,dates,window,mode):
    monthly=complete_months(frame).tail(window)
    x=np.arange(len(monthly))/12
    end=x[-1]
    horizon=(dates.year-monthly.index[-1].year)*12+dates.month-monthly.index[-1].month
    future=end+horizon.to_numpy()/12
    def basis(t,m):
        mm=np.eye(12)[m-1]
        if mode=='spline':
            return np.column_stack([mm,t-end,.55*np.maximum(t-end+1.5,0)**2,.55*np.maximum(t-end+3.5,0)**2])
        period=float(mode)
        return np.column_stack([mm,t-end,np.sin(2*np.pi*t/period),np.cos(2*np.pi*t/period)])
    model=Ridge(alpha=.25)
    model.fit(basis(x,monthly.index.month),np.log(monthly.clip(lower=1.0)))
    log_future=model.predict(basis(future,dates.month))
    last=model.predict(basis(np.repeat(end,len(dates)),dates.month))
    change=np.clip(log_future-last,-.34*(future-end)[:,None],.34*(future-end)[:,None])
    return np.exp(last+change)

def monthly_forecast(panel,dates,old,window,mode):
    q=panel['quantity'].to_numpy().sum()/panel['N'].to_numpy().sum()
    ratio=panel['actual'].sum(axis=1)/panel['base'].sum(axis=1)
    ratio=ratio.loc['2013':]
    factors=ratio.groupby([ratio.index.year%2,ratio.index.month,ratio.index.day]).median()
    factor=factors.reindex(pd.MultiIndex.from_arrays([dates.year%2,dates.month,dates.day])).fillna(1).to_numpy()
    out=[]
    for k,values in enumerate(['base','cost']):
        gross=panel[values].copy()
        gross['TOTAL']=gross.sum(axis=1)
        monthly=curve(gross,dates,window,mode)
        avg=.5*monthly[:,-1]+.5*monthly[:,:-1].sum(axis=1)
        discount=factor if k==0 else np.ones(len(dates))
        old_nominal=old[:,k]/discount
        by_month=pd.Series(old_nominal,index=dates).groupby(dates.to_period('M')).transform('mean').to_numpy()
        years=dates.to_period('M')
        for period,size in years.value_counts().items():
            if size<15:
                same_month=years==period-12
                if same_month.any():
                    by_month[years==period]=old_nominal[same_month].mean()
        out.append(q*avg*old_nominal/by_month*discount)
    return np.column_stack(out)

In [18]:
def turning_forecast(panel,dates,base):
    m=complete_months(panel['base'].sum(axis=1)).to_numpy()
    def change(n,lag):
        k=len(m)-lag
        return np.log(m[k-n:k].mean()/m[k-n-12:k-12].mean())
    short=change(3,0)
    earlier=change(6,6)
    annual=change(12,0)
    # Tăng trọng số đường dài khi xu hướng gần đây củng cố tín hiệu của năm.
    same=short*annual>0
    acceleration=(short-earlier)*np.sign(annual)
    trend_weight=.65*np.clip(acceleration/.18,0,1)*np.clip(abs(short)/.16,0,1) if same else 0.
    q=m[-24:].reshape(8,3).mean(axis=1)
    changes=np.log(q[-4:]/q[:4])
    consistency=np.mean(changes*np.sign(annual)>0)
    trend_weight*=np.clip(2*consistency-1,0,1)
    volume_month=complete_months(panel['N'].sum(axis=1)).to_numpy()
    vq=volume_month[-24:].reshape(8,3).mean(axis=1)
    volume_breadth=np.clip(2*np.mean(vq[4:]>vq[:4])-1,0,1)
    if annual>0:
        trend_weight*=volume_breadth
    spline=monthly_forecast(panel,dates,base,120,'spline')
    local=monthly_forecast(panel,dates,base,36,'7')
    # Phối hợp mô hình cơ sở với hai dự báo mức tháng.
    merged=(1-trend_weight)*base+trend_weight*spline
    result=.85*merged+.15*local
    # Hồi phục về mức dài hạn khi nhiều quý liên tiếp dịch chuyển cùng hướng.
    effects=[]
    for col in ['N','base']:
        month=complete_months(panel[col].sum(axis=1)).to_numpy()
        quarter=month[-24:].reshape(8,3).mean(axis=1)
        yoy=np.log(quarter[4:]/quarter[:4])
        gap=np.log(month[-96:-12].mean()/month[-12:].mean())
        signed=yoy*np.sign(gap)
        evidence=np.clip(signed.mean()/(2*max(.035,signed.std(ddof=1)/2)),0,1)
        breadth=np.clip(2*np.mean(signed>0)-1,0,1)
        effects.append(np.clip(gap,-1,1)*evidence*breadth)
    horizon=np.arange(1,len(dates)+1)/365.25
    effect=np.mean(effects)
    if effect>0:
        effect*=volume_breadth
    return result*np.exp(effect*(1-np.exp(-.75*horizon)))[:,None]

#### c. Phối hợp mùa vụ và CatBoost
Dự báo thành phần, mùa vụ lịch sử (tối đa 6 năm đầy đủ), CatBoost học log tỷ lệ ngày/trung bình tháng được phối hợp 65%–20%–15%. Khi validation sớm, dùng số năm đầy đủ thực sự có sẵn. Chỉ dùng đặc trưng lịch đã biết trước.

In [19]:
# Các hàm dự báo Ensemble chính — được dùng cả khi kiểm định và xuất file
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
def feat(idx):
 idx=pd.DatetimeIndex(idx); n=len(idx);m=idx.month.to_numpy();d=idx.day.to_numpy();dow=idx.dayofweek.to_numpy();e=idx.days_in_month.to_numpy()-d
 return pd.DataFrame(dict(month=m,day=d,dow=dow,end=e,quarter=idx.quarter,month_in_quarter=(m-1)%3,year_parity=idx.year%2,year_4=idx.year%4,dayofyear=idx.dayofyear,year=idx.year,month_day=m*32+d,
    is_wkend=(dow>=5).astype(int),week_in_month=((d-1)//7)+1,days_in_month=idx.days_in_month,
    is_month_start=(d==1).astype(int),is_month_end=(e==0).astype(int),
    day_sin=np.sin(2*np.pi*idx.dayofyear/365.25),day_cos=np.cos(2*np.pi*idx.dayofyear/365.25),
    dow_sin=np.sin(2*np.pi*dow/7),dow_cos=np.cos(2*np.pi*dow/7)))


def shape_cat(train,dates, iters=450,depth=5,history=8):
 # Data shape learned using only complete months.
 hist=train.loc[train.index >= train.index.max()-pd.DateOffset(years=history)].copy()
 groups=hist.index.to_period('M')
 valid=(pd.Series(np.ones(len(hist)),index=hist.index).groupby(groups).transform('size').to_numpy()==hist.index.days_in_month)
 hist=hist[valid];groups=hist.index.to_period('M')
 ratio=(hist/hist.groupby(groups).transform('mean')).clip(.07,9)
 X=feat(hist.index).drop(columns=['year']);Xt=feat(dates).drop(columns=['year']);out=[]
 wt=np.exp(-np.arange(len(hist))[::-1]/(365.25*5))
 for j,col in enumerate(['Revenue','COGS']):
  cb=CatBoostRegressor(iterations=iters,depth=depth,learning_rate=.045,loss_function='RMSE',l2_leaf_reg=8,
        random_seed=123,verbose=False,thread_count=2,allow_writing_files=False)
  cb.fit(X, np.log(ratio[col].to_numpy()),sample_weight=wt)
  out.append(np.exp(cb.predict(Xt)))
  del cb
 out=np.column_stack(out)
 # Normalize to preserve monthly means.
 m=pd.DataFrame(out,index=dates).groupby(dates.to_period('M')).transform('mean').to_numpy()
 return out/np.maximum(m,1e-8)


def pattern(train, dates, n=6):
    years = train.groupby(train.index.year).size()
    complete = [int(y) for y, count in years.items() if count == (366 if pd.Timestamp(f"{y}-01-01").is_leap_year else 365)]
    assert complete, "Không có năm lịch sử đầy đủ."
    d = train.loc[train.index.year.isin(complete[-n:])].copy()
    d = d / d.groupby(d.index.to_period('M')).transform('mean')
    key = dates.month.to_numpy()*100 + dates.day.to_numpy()
    key = np.where(key == 229, 228, key)
    profiles = [d.loc[d.index.year == yr].groupby(lambda x: x.month*100+x.day).mean().reindex(key).to_numpy() for yr in complete[-n:]]
    return np.mean(profiles, axis=0)

FACTOR_WEIGHT = 0.65
HISTORICAL_WEIGHT = 0.20
CATBOOST_WEIGHT = 0.15
assert abs(FACTOR_WEIGHT+HISTORICAL_WEIGHT+CATBOOST_WEIGHT-1)<1e-12

def blend_daily_shapes(past, dates, factor_prediction):
    dates = pd.DatetimeIndex(dates)
    assert past.index.max() < dates.min(), 'Có nguy cơ dùng dữ liệu tương lai'
    base = np.asarray(factor_prediction,dtype=float)
    m = pd.DataFrame(base,index=dates).groupby(dates.to_period('M')).transform('mean').to_numpy()
    hist_pattern = pattern(past,dates,n=6)
    ml_pattern = shape_cat(past,dates,iters=450,depth=5,history=8)
    pred = FACTOR_WEIGHT*base + m*(HISTORICAL_WEIGHT*hist_pattern + CATBOOST_WEIGHT*ml_pattern)
    return np.maximum(pred,1.)



#### d. Phục hồi có điều kiện
Phát hiện năm giảm mạnh từ các năm đầy đủ; chỉ điều chỉnh nếu mức giảm vượt 25% trong cửa sổ 8 năm. Cường độ 0,445, suy giảm theo hằng số 600 ngày, kích hoạt khi đa số tháng tăng trưởng cùng kỳ trong cả 6 và 12 tháng gần nhất.

In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from threadpoolctl import threadpool_limits

TARGETS = ['Revenue','COGS']

def recovery_adjustment(past, dates, raw_prediction, strength=0.50, tau=600):
    """Tổng quát hoá hàm Ensemble; strength=.50, tau=600 cho cùng kết quả với Ensemble."""
    dates=pd.DatetimeIndex(dates)
    assert past.index.max()<dates.min(), 'Không được dùng dữ liệu sau thời điểm dự báo'
    year_mean=past[TARGETS].groupby(past.index.year).mean()
    days=past.groupby(past.index.year).size()
    complete=year_mean.loc[days.loc[year_mean.index].to_numpy() == np.where(pd.to_datetime(year_mean.index.astype(str)+"-01-01").is_leap_year,366,365)]
    drop_year=int(complete.Revenue.pct_change().idxmin())
    drop=float(complete.Revenue.pct_change().loc[drop_year])
    if drop>=-0.25 or (dates.min().year-drop_year)>8:
        return np.asarray(raw_prediction).copy(), {'regime_detected':False,'drop':drop}
    anchor=complete.loc[drop_year-1, TARGETS].to_numpy(dtype=float)
    year_level=pd.DataFrame(raw_prediction,index=dates,columns=TARGETS).groupby(dates.year).transform('mean').to_numpy()
    w=strength*np.exp(-np.arange(len(dates))/float(tau))
    factor=1+w[:,None]*(anchor/np.maximum(year_level,1.)-1.)
    return np.maximum(np.asarray(raw_prediction)*factor,1.),{
        'regime_detected':True, 'drop_year':drop_year, 'anchor_year':drop_year-1,
        'strength':strength, 'tau':tau, 'anchor':anchor.tolist()}



In [21]:
def growth_gate(past):
    m = past[TARGETS].resample('MS').mean()
    counts = past.resample('MS').size()
    m = m.loc[counts == counts.index.days_in_month]
    yoy = m.Revenue.pct_change(12).dropna()
    assert len(yoy)>=12
    p6,p12 = float((yoy.tail(6)>0).mean()),float((yoy.tail(12)>0).mean())
    return (min(p6,p12) if min(p6,p12)>0.5 else 0.),p6,p12

def base_ensemble(hist,seg,dates):
    with threadpool_limits(limits=2):
        base=forecast_components(hist,dates).to_numpy()
        factor_prediction=turning_forecast(seg,dates,base)
        ensemble=blend_daily_shapes(hist[TARGETS],dates,factor_prediction)
        recovered,_=recovery_adjustment(hist,dates,ensemble,strength=.445,tau=600)
    gate,p6,p12=growth_gate(hist)
    return {'Ensemble':ensemble,'Recovered':recovered,'Gated':ensemble+gate*(recovered-ensemble)},gate,p6,p12

#### e. Hiệu chỉnh ngoài mẫu và dự báo thành phần dài hạn
Chỉ dùng sai số của các dự báo validation đã có nhãn trước origin. Lượt đầu chưa có bản ghi ngoài mẫu nên bỏ qua hiệu chỉnh. Lần lượt hiệu chỉnh ngày cuối tháng, Ridge theo ngày/thứ, nhánh xu hướng giảm dần rồi Ridge sai số ngoài mẫu. Hiệu chỉnh hình dạng bảo toàn tổng tháng dự báo.

In [22]:
def month_end_residual(records, dates, pred, strength=.15):
    dates=pd.DatetimeIndex(dates); chunks=[]
    for rec in records:
        dd=pd.DatetimeIndex(rec['dates']); mask=(dd<dates[0]) & np.isfinite(rec['actual']).all(axis=1)
        dd=dd[mask]; a=rec['actual'][mask]; p=rec['ensemble'][mask]
        if not len(dd): continue
        grp=dd.to_period('M'); full=pd.Series(1,index=dd).groupby(grp).transform('size').to_numpy()==dd.days_in_month
        dd=dd[full];a=a[full];p=p[full]
        if not len(dd): continue
        grp=dd.to_period('M');a=a/pd.DataFrame(a).groupby(grp).transform('mean').to_numpy();p=p/pd.DataFrame(p).groupby(grp).transform('mean').to_numpy()
        chunks.append(pd.DataFrame(np.log(a/p),index=dd,columns=['Revenue','COGS']))
    if not chunks: return pred.copy()
    residual=pd.concat(chunks).groupby(level=0).mean(); end=residual.index.days_in_month-residual.index.day
    profile=residual.groupby(end).median();corr=profile.reindex(dates.days_in_month-dates.day).fillna(0).to_numpy();q=pred*np.exp(strength*np.clip(corr,-.3,.3));g=dates.to_period('M');q*=pd.DataFrame(pred).groupby(g).transform('sum').to_numpy()/pd.DataFrame(q).groupby(g).transform('sum').to_numpy();return q

In [23]:
def residual_data(records, origin):
 chunks=[]
 for r in records:
  d=pd.DatetimeIndex(r['dates']);mask=(d<origin)&np.isfinite(r['actual']).all(1);d=d[mask];a=r['actual'][mask];p=r['ensemble'][mask];g=d.to_period('M');full=pd.Series(1,index=d).groupby(g).transform('size').values==d.days_in_month;d=d[full];a=a[full];p=p[full];g=d.to_period('M');
  if not len(d):continue
  a=a/pd.DataFrame(a).groupby(g).transform('mean').values;p=p/pd.DataFrame(p).groupby(g).transform('mean').values
  chunks.append(pd.DataFrame(np.log(a/p),index=d,columns=TARGETS))
 return pd.concat(chunks).groupby(level=0).mean() if chunks else pd.DataFrame(columns=TARGETS,index=pd.DatetimeIndex([]))

def features(d):
 d=pd.DatetimeIndex(d);return pd.DataFrame({'month':d.month,'day':d.day,'end':d.days_in_month-d.day,'dow':d.dayofweek,'parity':d.year%2,'week':(d.day-1)//7,'quarter':d.quarter,'startbin':np.minimum(d.day,8),'endbin':np.minimum(d.days_in_month-d.day,8)},index=d)
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

def correction(records,d,p,combo,alpha,strength,decay=0):
 d=pd.DatetimeIndex(d);r=residual_data(records,d[0]);
 if len(r)<120:return p.copy()
 tr=features(r.index);te=features(d);X=[];Xt=[]
 for cols in combo:
  enc=OneHotEncoder(handle_unknown='ignore');a=tr[list(cols)].astype(str).agg('_'.join,axis=1).values[:,None];b=te[list(cols)].astype(str).agg('_'.join,axis=1).values[:,None];X.append(enc.fit_transform(a));Xt.append(enc.transform(b))
 w=np.exp(-(d[0]-r.index).days.values/decay) if decay else None
 model=Ridge(alpha=alpha,solver='lsqr').fit(hstack(X).tocsr(),r.values,sample_weight=w);c=model.predict(hstack(Xt).tocsr());q=p*np.exp(strength*np.clip(c,-.4,.4));g=d.to_period('M');q*=pd.DataFrame(p).groupby(g).transform('sum').values/pd.DataFrame(q).groupby(g).transform('sum').values;return q


def calendar_residual(records, dates, pred):
    return correction(records, dates, pred, [("day",), ("dow",)], alpha=10, strength=.2, decay=365)


In [24]:
def factor_model(hist, dates, window=36, damp=12, alpha=1, recent=24):
 m=hist[['lines','Revenue','COGS']].resample('MS').sum(); counts=hist.resample('MS').size();m=m.loc[counts==counts.index.days_in_month];m['Revenue']/=m.lines;m['COGS']/=m.lines;m['lines']/=m.index.days_in_month
 m=m.tail(window);t=np.arange(len(m))-len(m)+1
 d=pd.DatetimeIndex(dates);future=(d.year-m.index[-1].year)*12+d.month-m.index[-1].month
 def basis(idx,t):
  month=idx.month.to_numpy();aug=(month==8)*(idx.year%2*2-1);return np.column_stack([t/12,np.eye(12)[month-1],aug])
 x=basis(m.index,t); y=np.log(m.to_numpy()); w=.5**(-t/recent); reg=Ridge(alpha=alpha);reg.fit(x,y,sample_weight=w)
 ft=damp*(1-np.exp(-future.to_numpy()/damp)) if damp else future.to_numpy()
 raw=np.exp(reg.predict(basis(d,ft)))
 return raw[:,0,None]*raw[:,1:]
def level_ratio(hist,d,p,window,damp,alpha,recent,mode):
 a=factor_model(hist,d,window,damp,alpha,recent);g=d.to_period('M');b=pd.DataFrame(p).groupby(g).transform('mean').values
 for per in g.unique():
  mask=g==per
  if mask.sum()<per.days_in_month and (g==per-12).any():b[mask]=p[g==per-12].mean(0)
 if mode=='level':a=a/a.mean(0)*b.mean(0)
 return np.clip(np.log(a/b),-.7,.7)

def damped_factor_adjustment(hist,dates,p):
    # Fixed conservative blend of an independent factor forecast; preserve the baseline overall mean in the alternative monthly level.
    return p*np.exp(.2*level_ratio(hist,dates,p,60,12,.1,24,'level'))


In [25]:
def new_resid(records,d,p,cols,alpha,weight,mode,half):
 parts=[]
 for _,dd,pp,yy in records:
  mask=(dd<d[0])&np.isfinite(yy).all(1);dd=dd[mask];pp=pp[mask];yy=yy[mask];g=dd.to_period('M');valid=pd.Series(1,index=dd).groupby(g).transform('sum').values==dd.days_in_month;dd=dd[valid];pp=pp[valid];yy=yy[valid];g=dd.to_period('M')
  if not len(dd):continue
  if mode=='shape':
   pp=pp/pd.DataFrame(pp).groupby(g).transform('mean').values;yy=yy/pd.DataFrame(yy).groupby(g).transform('mean').values
  val=np.log(yy/pp);parts.append(pd.DataFrame(val,index=dd))
 if not parts:return p.copy()
 rr=pd.concat(parts).groupby(level=0).mean();
 if len(rr)<120:return p.copy()
 def feat(d):return pd.DataFrame({'m':d.month,'d':d.day,'e':d.days_in_month-d.day,'w':d.dayofweek,'p':d.year%2,'q':d.year%4,'n':d.days_in_month,'b':(d.day-1)//7,'y':d.year,'early':np.minimum(d.day,5),'late':np.minimum(d.days_in_month-d.day,5)})
 a=feat(rr.index);b=feat(d);xs=[];xt=[]
 for c in cols:
  enc=OneHotEncoder(handle_unknown='ignore');x=a[list(c)].astype(str).agg('_'.join,axis=1).values[:,None];xx=b[list(c)].astype(str).agg('_'.join,axis=1).values[:,None];xs.append(enc.fit_transform(x));xt.append(enc.transform(xx))
 w=.5**((d[0]-rr.index).days.to_numpy()/half) if half else None
 reg=Ridge(alpha=alpha,solver='lsqr').fit(hstack(xs).tocsr(),rr.values,sample_weight=w);c=reg.predict(hstack(xt).tocsr());q=p*np.exp(weight*np.clip(c,-.3,.3));g=d.to_period('M')
 if mode=='shape':q*=pd.DataFrame(p).groupby(g).transform('sum').values/pd.DataFrame(q).groupby(g).transform('sum').values
 return q


def oof_shape_adjustment(records, dates, pred):
    return new_resid(records, dates, pred, (("m",),("w",),("early",),("late",)), 20, .2, "shape", 730)


#### f. Chu kỳ hai năm và dự báo tám phân khúc
Ridge + Fourier tương tác chẵn/lẻ, tối đa 84 tháng, alpha 0,3, half-life 48 tháng, damping 24 tháng. Nhánh tổng thể phối hợp 15% trên log tỷ lệ, nhánh phân khúc dự báo số dòng/ngày, Revenue/dòng, COGS/dòng rồi tổng hợp tám nhóm; phối hợp 30% trên log tỷ lệ sau chuẩn hóa mức theo năm hoặc phần năm dự báo.

In [26]:
def monthly(hist):
 m=hist[['lines','Revenue','COGS']].resample('MS').sum();cnt=hist.resample('MS').size();m=m.loc[cnt==cnt.index.days_in_month];m['Revenue']/=m.lines;m['COGS']/=m.lines;m['lines']/=m.index.days_in_month;return m

def alternative(hist,d,window=60,alpha=2,half=36,damp=12,parity=1):
 m=monthly(hist).tail(window);t=np.arange(len(m))-len(m)+1;last=m.index[-1];hf=(d.year-last.year)*12+d.month-last.month;ft=damp*(1-np.exp(-hf/damp))
 def basis(idx,t):
  month=idx.month.to_numpy(); parityv=idx.year.to_numpy()%2*2-1; a=2*np.pi*(month-1)/12; seasonal=np.column_stack([np.sin(a),np.cos(a),np.sin(2*a),np.cos(2*a),np.sin(3*a),np.cos(3*a)]);return np.column_stack([t/12,np.eye(12)[month-1],parityv[:,None]*seasonal*parity,parityv*.5,(month==8)*parityv])
 reg=Ridge(alpha=alpha).fit(basis(m.index,t),np.log(m.values),sample_weight=.5**(-t/half));v=np.exp(reg.predict(basis(d,ft)));return v[:,0,None]*v[:,1:]

def apply_alt(h,d,p,a,w,mode):
 g=d.to_period('M');b=pd.DataFrame(p).groupby(g).transform('mean').values
 for per in g.unique():
  mask=g==per
  if mask.sum()<per.days_in_month and (g==per-12).any():b[mask]=p[g==per-12].mean(0)
 if mode=='level':a=a/a.mean(0)*b.mean(0)
 elif mode=='year':
  a=a.copy()
  for yy in np.unique(d.year):
   mask=d.year==yy;a[mask]=a[mask]/a[mask].mean(0)*b[mask].mean(0)
 return p*np.exp(w*np.clip(np.log(a/b),-.5,.5))


def biennial_adjustment(hist,d,p):
    return apply_alt(hist,d,p,alternative(hist,d,84,.3,48,24,.5),.15,'year')


In [27]:
def segment_alternative(h,d,segmonthly,window=84,alpha=2,half=48,damp=24):
 m=segmonthly.loc[segmonthly.index + pd.offsets.MonthEnd(0) <= h.index.max()].tail(window);t=np.arange(len(m))-len(m)+1;idx=m.index;f=(d.year-idx[-1].year)*12+d.month-idx[-1].month;ft=damp*(1-np.exp(-f/damp));
 def design(idx,t):
  mo=idx.month.to_numpy();pa=idx.year.to_numpy()%2*2-1;rad=2*np.pi*(mo-1)/12;sea=np.column_stack([np.sin(rad),np.cos(rad),np.sin(2*rad),np.cos(2*rad)]);return np.column_stack([t/12,np.eye(12)[mo-1],pa*.5,pa[:,None]*sea*.5,(mo==8)*pa])
 count=m['lines'].values;v=np.concatenate([count/idx.days_in_month.to_numpy()[:,None],m['Revenue'].values/count,m['COGS'].values/count],axis=1);reg=Ridge(alpha=alpha).fit(design(idx,t),np.log(v),sample_weight=.5**(-t/half));q=np.exp(reg.predict(design(d,ft)));k=count.shape[1];return np.column_stack([(q[:,:k]*q[:,k:2*k]).sum(1),(q[:,:k]*q[:,2*k:]).sum(1)])

def segment_adjustment(hist,d,p,segmonthly):
 return apply_alt(hist,d,p,segment_alternative(hist,d,segmonthly,84,.3,48,24),.3,'year')


In [28]:
records = []
factor_records = []
hybrid_diagnostics = []

def hybrid_forecast(train, dates, return_details=False):
    dates = pd.DatetimeIndex(dates)
    origin = dates.min()
    hist = daily.loc[:train.index.max()].copy()
    segments = panel.loc[:train.index.max()].copy()
    segment_months = complete_months(seg_daily.loc[:train.index.max()], how="sum")
    assert hist.index.equals(train.index) and segments.index.equals(train.index)
    assert hist.index.max() < origin and origin == hist.index.max() + pd.Timedelta(days=1)
    assert (segment_months.index + pd.offsets.MonthEnd(0) < origin).all()
    for record in records:
        assert pd.DatetimeIndex(record["dates"]).max() < origin
    for _, dd, _, _ in factor_records:
        assert pd.DatetimeIndex(dd).max() < origin
    with threadpool_limits(limits=2):
        base, gate, p6, p12 = base_ensemble(hist, segments, dates)
        a = month_end_residual(records, dates, base["Gated"])
        b = calendar_residual(records, dates, a)
        c = damped_factor_adjustment(hist, dates, b)
        d = oof_shape_adjustment(factor_records, dates, c)
        e = biennial_adjustment(hist, dates, d)
        q = segment_adjustment(hist, dates, e, segment_months)
    assert q.shape == (len(dates), 2) and np.isfinite(q).all() and (q > 0).all()
    result = pd.DataFrame(q, index=dates, columns=targets)
    detail = {"ensemble":base["Ensemble"], "factor_level":c, "gate":gate, "p6":p6, "p12":p12, "without_segment":e, "calibration_folds":len(records)}
    return (result, detail) if return_details else result


### 6.12. Chạy thực nghiệm và tạo dự báo
Hybrid Ensemble là mô hình dự báo chính cho Revenue và COGS. So sánh 11 phương pháp qua ba lượt validation và ghi nhận phương pháp có MAE trung bình thấp nhất của từng chỉ tiêu để làm đối chứng. Tạo dự báo test cho Hybrid, baseline, hai mô hình mùa vụ và phương pháp đứng đầu validation. Chỉ tính chỉ số test từ notebook 03.

In [29]:
records.clear()
factor_records.clear()
hybrid_diagnostics.clear()

predictors = {
    "Naive": predict_naive,
    "Trung bình 28 ngày": predict_mean_28,
    "Theo thứ trong tuần": predict_weekday,
    "ETS (A,N,N)": predict_ets,
    "ARIMA(1,1,1)": predict_arima,
    "Hồi quy tuyến tính": predict_trend,
    "Hồi quy tuyến tính + mùa vụ": predict_trend_with_seasonality,
    "Baseline cuộc thi": predict_contest_baseline,
    "Mùa vụ, mức gốc cố định": predict_fixed_level_seasonal,
    "Mùa vụ mức gốc cố định + hiệu ứng thứ": predict_fixed_level_with_weekday,
    hybrid_name: hybrid_forecast,
}

def evaluate(actual, forecast):
    pd.testing.assert_index_equal(actual.index, forecast.index)
    pd.testing.assert_index_equal(actual.columns, forecast.columns)
    if not np.isfinite(forecast.to_numpy(dtype=float)).all():
        raise ValueError("Dự báo có giá trị thiếu hoặc không hữu hạn.")
    actual = actual.astype(float) / 1_000_000
    error = forecast.astype(float) / 1_000_000 - actual
    squared_error = error.pow(2)
    variation = (actual - actual.mean()).pow(2).sum()
    return pd.DataFrame({
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt(squared_error.mean()),
        "R²": 1 - squared_error.sum() / variation.where(variation > 0),
    }).rename_axis("Chỉ tiêu")

from tempfile import TemporaryDirectory

prediction_parts = []
score_rows = []
timing_rows = []


def prediction_block(forecast, split, fold_id, train_end, method):
    return forecast.rename_axis("Date").reset_index().assign(
        Split=split, Fold=fold_id, TrainEnd=train_end, Method=method,
    )


for _, fold in folds.iterrows():
    fold_id = int(fold["Fold"])
    train_end = pd.Timestamp(fold["Train đến"])
    train = sales_dev.loc[:train_end, targets]
    actual = sales_dev.loc[pd.Timestamp(fold["Validation từ"]):pd.Timestamp(fold["Validation đến"]), targets]

    for method, predictor in predictors.items():
        started = perf_counter()
        if method == hybrid_name:
            forecast, detail = hybrid_forecast(train, actual.index, return_details=True)
            hybrid_diagnostics.append({"Fold": fold_id, "TrainEnd": train_end, "Recovery gate": detail["gate"], "OOF folds": detail["calibration_folds"]})
            records.append({"dates": actual.index.to_numpy(), "actual": actual.to_numpy().copy(), "ensemble": detail["ensemble"]})
            factor_records.append((str(train_end.date()), actual.index, detail["factor_level"], actual.to_numpy().copy()))
        else:
            forecast = predictor(train, actual.index)
        timing_rows.append({"Phương pháp": method, "Giây/lượt": perf_counter() - started})
        score_rows.append(evaluate(actual, forecast).assign(Method=method, Fold=fold_id))
        prediction_parts.append(prediction_block(forecast, "validation", fold_id, train_end, method))

display(pd.DataFrame(hybrid_diagnostics))
validation_scores = pd.concat(score_rows).reset_index()
display(validation_scores.groupby(["Method", "Chỉ tiêu"])[["MAE", "RMSE", "R²"]].mean().unstack("Chỉ tiêu").reindex(predictors).round(3))
mean_mae = pd.concat(score_rows).reset_index().groupby(["Method", "Chỉ tiêu"])["MAE"].mean()
validation_best = mean_mae.unstack("Chỉ tiêu").idxmin().reindex(targets)
display(validation_best.rename("MAE validation thấp nhất"))
display(pd.DataFrame(timing_rows).groupby("Phương pháp").mean().reindex(predictors).round(3))

final_train = sales_dev[targets]
final_dates = pd.DatetimeIndex(test_dates)
assert final_train.index.max() + pd.Timedelta(days=1) == final_dates.min()
test_methods = list(dict.fromkeys([contest_name, fixed_name, weekday_name, hybrid_name, *validation_best.unique()]))
test_forecasts = {method: predictors[method](final_train, final_dates) for method in test_methods}
validation_reference = pd.DataFrame(index=final_dates, columns=targets, dtype=float)
for target, method in validation_best.items():
    validation_reference[target] = test_forecasts[method][target]
for method, forecast in test_forecasts.items():
    prediction_parts.append(prediction_block(forecast, "test", 0, final_train.index.max(), method))
prediction_parts.append(prediction_block(validation_reference, "test", 0, final_train.index.max(), validation_csv_label))
predictions = pd.concat(prediction_parts, ignore_index=True)[
    ["Split", "Fold", "TrainEnd", "Date", "Method", *targets]
]

assert not predictions.duplicated(["Split", "Fold", "Method", "Date"]).any()
assert predictions.groupby(["Split", "Fold", "Method"]).size().eq(horizon).all()
assert np.isfinite(predictions[targets].to_numpy()).all()
assert (predictions["Date"] > predictions["TrainEnd"]).all()
prediction_path = project_root / "data/model/du_bao/forecast_predictions.csv"
prediction_path.parent.mkdir(parents=True, exist_ok=True)
csv_text = predictions.to_csv(index=False, date_format="%Y-%m-%d", lineterminator=chr(10))
if not prediction_path.is_file() or prediction_path.read_text(encoding="utf-8") != csv_text:
    with TemporaryDirectory(dir=prediction_path.parent) as directory:
        temporary_path = Path(directory) / prediction_path.name
        temporary_path.write_bytes(csv_text.encode("utf-8"))
        temporary_path.replace(prediction_path)
print(f"Đã xác nhận {len(predictions):,} dòng dự báo tại {prediction_path.relative_to(project_root)}.")

,Fold,TrainEnd,Recovery gate,OOF folds
0,1,2016-12-30,0.666667,0
1,2,2018-07-01,0.000000,1
2,3,2019-12-31,0.000000,2


MAE           RMSE             R²  \
Chỉ tiêu                                COGS Revenue   COGS Revenue   COGS   
Method                                                                       
Naive                                  3.039   3.124  3.404   3.603 -3.967   
Trung bình 28 ngày                     2.504   3.091  3.064   3.784 -2.268   
Theo thứ trong tuần                    2.525   3.106  3.103   3.819 -2.357   
ETS (A,N,N)                            3.041   3.125  3.405   3.603 -3.968   
ARIMA(1,1,1)                           2.605   3.171  3.121   3.811 -2.683   
Hồi quy tuyến tính                     2.138   2.460  2.510   2.905 -0.919   
Hồi quy tuyến tính + mùa vụ            1.617   1.858  1.910   2.228 -0.337   
Baseline cuộc thi                      1.195   1.375  1.485   1.727  0.195   
Mùa vụ, mức gốc cố định                1.048   1.212  1.324   1.548  0.374   
Mùa vụ mức gốc cố định + hiệu ứng thứ  1.018   1.182  1.304   1.530  0.384   
Hybrid Ensemble phân rã và phân khúc   1.131   1.299  1.443   1.667  0.031   

                                               
Chỉ tiêu                              Revenue  
Method                                         
Naive                                  -2.674  
Trung bình 28 ngày                     -2.471  
Theo thứ trong tuần                    -2.545  
ETS (A,N,N)                            -2.674  
ARIMA(1,1,1)                           -2.865  
Hồi quy tuyến tính                     -0.827  
Hồi quy tuyến tính + mùa vụ            -0.285  
Baseline cuộc thi                       0.256  
Mùa vụ, mức gốc cố định                 0.414  
Mùa vụ mức gốc cố định + hiệu ứng thứ   0.422  
Hybrid Ensemble phân rã và phân khúc    0.086

Chỉ tiêu
Revenue    Mùa vụ mức gốc cố định + hiệu ứng thứ
COGS       Mùa vụ mức gốc cố định + hiệu ứng thứ
Name: MAE validation thấp nhất, dtype: object

,Giây/lượt
Phương pháp,
Naive,0.000
Trung bình 28 ngày,0.001
Theo thứ trong tuần,0.005
"ETS (A,N,N)",0.046
"ARIMA(1,1,1)",0.912
Hồi quy tuyến tính,0.005
Hồi quy tuyến tính + mùa vụ,0.012
Baseline cuộc thi,0.023
"Mùa vụ, mức gốc cố định",0.026


Đã xác nhận 20,824 dòng dự báo tại data\model\du_bao\forecast_predictions.csv.


**Giới hạn:** giữ kiến trúc và tham số tham chiếu, nhưng sửa kỳ không đầy đủ và giới hạn bản ghi ngoài mẫu. Đây là triển khai hồi cứu thích nghi, không phải tái tạo nguyên xi một kỳ dự báo khác. Các chỉ số test và đóng góp thành phần sẽ đánh giá ở notebook 03 và 04.

**Kết quả validation (triệu đơn vị tiền/ngày).** Mùa vụ mức gốc cố định + hiệu ứng thứ có MAE trung bình thấp nhất cho Revenue (1,181767) và COGS (1,018286). Hybrid đạt lần lượt 1,298505 và 1,131490. Notebook 03 sử dụng Hybrid cho đầu ra dự báo chính và dùng kết quả validation để đối chiếu.